# 03 매체그룹 비교 — 의제 비중(soft topic mass) + 단어 선택(log-odds)

02의 `문서토픽분포_*.csv`(theta·dominant·high_purity)와 01의 `분석코퍼스_*.csv`(tokens)로 4매체그룹 비교

- **의제 비중 주지표 = 그룹별 평균 topic mass(soft, theta 가중) + 문서 부트스트랩 CI** — 문서 안 버림. dominant 비율은 맥락 보조
- 비중표 2버전 — (a) 우주 N 대비 절대, (b) 핵심의제 내부 조건부
- 단어 선택 = **같은 핵심의제 내부에서만 log-odds(Monroe, Dirichlet prior, z-score 주지표)**, 셀=그룹×의제, 이중기준 ≥50문서 AND ≥5,000토큰. hard 라벨 기반이라 **보조적 어휘차이 분석**으로 격하(주증거는 topic mass)
- 불확실성 = **문서 부트스트랩(press 집합 고정, 각 press 내부 문서만 복원추출)** — 그룹당 매체 2~3개라 매체 단위 재표집 금지. CI는 기술적 변동폭. LOPO·개별 매체는 강건성 패널(CI 아님)


In [ ]:
# from google.colab import drive; drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os, re, unicodedata
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

try:
    PROJECT_DIR = Path('/content/drive/MyDrive/Text-data-Analysis_26-Spring')
    if not PROJECT_DIR.exists(): raise FileNotFoundError
except Exception:
    PROJECT_DIR = Path('/home/carol/Text-data-Analysis_26-Spring')
os.chdir(PROJECT_DIR)
RESULT_DIR = PROJECT_DIR / 'outputs'
GROUPS = ['경제','통신·보도','정치색','지상파']

# 한글 폰트 폴백 — 환경별로 설치된 것 자동 선택
for cand in ['NanumGothic','Malgun Gothic','AppleGothic','NanumGothicCoding']:
    if any(cand.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = cand; break
plt.rcParams['axes.unicode_minus'] = False

def normalize_name(p): return unicodedata.normalize('NFC', p.name)
def find_one(pat):
    c = sorted(p for p in RESULT_DIR.iterdir() if p.is_file() and re.match(pat, normalize_name(p)))
    if not c: raise FileNotFoundError(pat+' 없음 — 앞 단계 먼저 실행')
    if len(c) > 1: print('⚠ 여러 기간 파일 —', normalize_name(c[-1]), '사용')
    return c[-1]

DT_PATH = find_one(r'^문서토픽분포_언론사_\d{6}_\d{6}\.csv$')
CORP_PATH = find_one(r'^분석코퍼스_언론사_\d{6}_\d{6}\.csv$')
TOPIC_PATH = find_one(r'^토픽_대표단어_언론사_\d{6}_\d{6}\.csv$')
PERIOD = re.search(r'(\d{6}_\d{6})', normalize_name(DT_PATH)).group(1)


In [ ]:
# --- 로드 + 토큰 join ---
dt = pd.read_csv(DT_PATH, encoding='utf-8-sig')
corp = pd.read_csv(CORP_PATH, encoding='utf-8-sig', usecols=['article_id','tokens'])
topics = pd.read_csv(TOPIC_PATH, encoding='utf-8-sig')
df = dt.merge(corp, on='article_id', how='left', validate='one_to_one')
assert df['tokens'].notna().all(), '토큰 join 결측'

THETA_COLS = sorted([c for c in df.columns if re.match(r'^theta_\d+$', c)], key=lambda c: int(c.split('_')[1]))  # 숫자 정렬 — 위치 j == 토픽 j 보장
K = len(THETA_COLS)
print('문서', len(df), '/ K', K, '/ 그룹', df['media_group'].value_counts().to_dict())
print('그룹별 매체 수(부트스트랩 군집·강건성 한계):', df.groupby('media_group')['press'].nunique().reindex(GROUPS).to_dict())

# 핵심 의제 = topic mass 상위 2~4 (사람이 CORE_TOPICS로 확정·라벨링)
topics = topics.sort_values('topic_mass', ascending=False)
print('topic mass 순:'); print(topics.to_string(index=False))
CORE_TOPICS = topics['topic'].head(4).tolist()   # ← 라벨링 후 수동 조정 가능(상위 2~4)
TOPIC_LABEL = {int(t): f'T{int(t)}' for t in topics['topic']}   # ← 사람 라벨로 교체
print('핵심 의제(잠정):', CORE_TOPICS)
print('⚠ CORE_TOPICS·TOPIC_LABEL은 잠정(top-4·플레이스홀더) — 토픽 대표단어 확인 후 사람이 핵심의제 2~4개 확정·라벨링하고 재실행')


In [ ]:
# --- 의제 비중 주지표 = 그룹별 평균 topic mass(soft) + 문서 부트스트랩 CI ---
# 부트스트랩: press 집합 고정, 각 press 내부 문서만 복원추출(매체 단위 재표집 금지)
theta = df[THETA_COLS].to_numpy()
grp = df['media_group'].to_numpy()
press = df['press'].to_numpy()
press_idx = {p: np.where(press == p)[0] for p in np.unique(press)}

def group_mass(rows):
    out = {}
    sub_g = grp[rows]; sub_t = theta[rows]
    for g in GROUPS:
        m = sub_g == g
        out[g] = sub_t[m].mean(axis=0) if m.any() else np.full(K, np.nan)
    return out   # g -> (K,) 평균 theta

point = group_mass(np.arange(len(df)))
B = 1000; rng = np.random.default_rng(0)
boot = {g: np.zeros((B, K)) for g in GROUPS}
for b in range(B):
    rows = np.concatenate([rng.choice(idx, size=len(idx), replace=True) for idx in press_idx.values()])
    gm = group_mass(rows)
    for g in GROUPS: boot[g][b] = gm[g]
ci_lo = {g: np.nanpercentile(boot[g], 2.5, axis=0) for g in GROUPS}
ci_hi = {g: np.nanpercentile(boot[g], 97.5, axis=0) for g in GROUPS}

mass_tbl = pd.DataFrame({g: point[g] for g in GROUPS}, index=[f'T{j}' for j in range(K)])
print('그룹별 평균 topic mass(주지표):'); print(mass_tbl.round(4))


In [ ]:
# --- 비중표 2버전 + dominant 비율(보조) + 그룹별 uncertain율 ---
# (a) 절대: 그룹 전체(우주) 대비 = 평균 topic mass 그대로
abs_share = mass_tbl.copy()
# (b) 조건부: 핵심의제들 내부에서만 재정규화
core_cols = [f'T{int(t)}' for t in CORE_TOPICS]
cond_share = abs_share.loc[core_cols] / abs_share.loc[core_cols].sum(axis=0)
print('(a) 절대 비중(핵심의제):'); print(abs_share.loc[core_cols].round(4))
print('(b) 조건부 비중(핵심의제 내부):'); print(cond_share.round(4))

# dominant 비율(맥락 보조) — 고순도(high_purity) 문서 기준
hp = df[df['high_purity']]
dom_share = (hp.groupby('media_group')['dominant_topic'].value_counts(normalize=True)
              .unstack(fill_value=0).reindex(GROUPS))
print('dominant 비율(고순도, 보조):'); print(dom_share.round(3))
unc_rate = (1 - df.groupby('media_group')['high_purity'].mean()).mul(100).round(1).reindex(GROUPS)
print('그룹별 uncertain(hard 비선정) 비율 %:'); print(unc_rate)


In [ ]:
# --- 단어 선택: 같은 핵심의제 내부 Monroe log-odds(Dirichlet prior) + z-score ---
# 멤버십 = 고순도 hard 문서 & dominant==해당 의제. 셀 = 그룹×의제. 이중기준 ≥50문서 AND ≥5,000토큰
from collections import Counter
def counts_of(rows_tokens):
    c = Counter()
    for s in rows_tokens: c.update(str(s).split())
    return c

# 배경 prior = 우주 전체 단어분포(Monroe informative prior)
bg = counts_of(df['tokens']); bg_total = sum(bg.values()); A0 = 1000.0
def alpha(w): return A0 * bg.get(w, 0) / bg_total

def logodds_z(cnt_g, cnt_rest):
    n_g = sum(cnt_g.values()); n_r = sum(cnt_rest.values())
    vocab = set(cnt_g) | set(cnt_rest)
    a0 = A0
    out = []
    for w in vocab:
        aw = alpha(w)
        yg, yr = cnt_g.get(w,0), cnt_rest.get(w,0)
        og = (yg+aw)/(n_g+a0-yg-aw); orr = (yr+aw)/(n_r+a0-yr-aw)
        delta = np.log(og) - np.log(orr)
        var = 1.0/(yg+aw) + 1.0/(yr+aw)
        out.append((w, delta/np.sqrt(var), yg))
    return sorted(out, key=lambda x: -x[1])

MIN_DOCS, MIN_TOKS = 50, 5000
rows_lo = []; cell_report = []
for t in CORE_TOPICS:
    mem = df[df['high_purity'] & (df['dominant_topic'] == t)]
    for g in GROUPS:
        gdocs = mem[mem['media_group'] == g]
        rdocs = mem[mem['media_group'] != g]
        ntok = int(gdocs['tokens'].map(lambda s: len(str(s).split())).sum())
        passed = (len(gdocs) >= MIN_DOCS) and (ntok >= MIN_TOKS)
        tier = ('main' if (passed and len(gdocs) >= 80) else ('appendix' if passed else 'none'))  # 50-79=부록, 80+=본문
        cell_report.append(dict(topic=int(t), group=g, n_docs=len(gdocs), n_tokens=ntok, passed=passed, tier=tier))
        if not passed: continue
        z = logodds_z(counts_of(gdocs['tokens']), counts_of(rdocs['tokens']))
        for w, zz, yg in z[:15]:
            rows_lo.append(dict(topic=int(t), group=g, word=w, z=round(zz,3), count=yg))
cell_df = pd.DataFrame(cell_report)
print('셀(그룹×의제) 이중기준 발동표:'); print(cell_df.to_string(index=False))
lo_df = pd.DataFrame(rows_lo)


In [ ]:
# --- 강건성 패널: 개별 매체 log-odds 점(상위 단어), CI 아님 ---
# 그룹당 매체 2~3개라 매체별 z를 점으로 나란히 — 특정 매체·사건명 잠식 점검
# (주의: 매체 vs 같은 의제 他매체 대조라 그룹-vs-pooled 본추정량과 동일하지 않은 진단용 점)
panel = []
for t in CORE_TOPICS:
    mem = df[df['high_purity'] & (df['dominant_topic'] == t)]
    for g in GROUPS:
        gdocs = mem[mem['media_group'] == g]
        if len(gdocs) < MIN_DOCS: continue
        for pr in sorted(gdocs['press'].unique()):
            pdocs = gdocs[gdocs['press'] == pr]; odocs = mem[mem['press'] != pr]
            if len(pdocs) < 20: continue
            z = logodds_z(counts_of(pdocs['tokens']), counts_of(odocs['tokens']))
            for w, zz, yg in z[:10]:
                panel.append(dict(topic=int(t), group=g, press=pr, word=w, z=round(zz,3)))
panel_df = pd.DataFrame(panel)
print('강건성 패널(매체별 상위 변별어) 행수:', len(panel_df))


In [ ]:
# --- 막대그래프(본증거) + 저장 ---
# 핵심의제별 그룹 평균 topic mass + 부트스트랩 CI
fig, ax = plt.subplots(figsize=(1.6*len(CORE_TOPICS)+3, 4))
x = np.arange(len(CORE_TOPICS)); w = 0.2
for i, g in enumerate(GROUPS):
    vals = [point[g][int(t)] for t in CORE_TOPICS]
    lo = [point[g][int(t)] - ci_lo[g][int(t)] for t in CORE_TOPICS]
    hi = [ci_hi[g][int(t)] - point[g][int(t)] for t in CORE_TOPICS]
    ax.bar(x + (i-1.5)*w, vals, w, yerr=[lo, hi], capsize=3, label=g)
ax.set_xticks(x); ax.set_xticklabels([TOPIC_LABEL[int(t)] for t in CORE_TOPICS])
ax.set_ylabel('평균 topic mass'); ax.set_title('핵심의제별 매체그룹 의제 비중(soft, 95% 부트스트랩 CI)')
ax.legend(); fig.tight_layout()
fig.savefig(RESULT_DIR / f'의제비중_막대_{PERIOD}.png', dpi=150)
print('그림 저장: 의제비중_막대')

abs_share.to_csv(RESULT_DIR / f'의제비중_절대_{PERIOD}.csv', encoding='utf-8-sig')
cond_share.to_csv(RESULT_DIR / f'의제비중_조건부_{PERIOD}.csv', encoding='utf-8-sig')
cell_df.to_csv(RESULT_DIR / f'logodds_셀발동표_{PERIOD}.csv', index=False, encoding='utf-8-sig')
if len(lo_df): lo_df.to_csv(RESULT_DIR / f'그룹특화어_logodds_{PERIOD}.csv', index=False, encoding='utf-8-sig')
if len(panel_df): panel_df.to_csv(RESULT_DIR / f'강건성_매체별특화어_{PERIOD}.csv', index=False, encoding='utf-8-sig')
print('저장: 의제비중_절대/조건부, logodds_셀발동표, 그룹특화어_logodds, 강건성_매체별특화어')
# ★ 핵심의제 라벨 — 04 감성과 공유(03에서 사람이 확정한 CORE_TOPICS·라벨을 04가 그대로 읽게 해 타깃 일치)
core_lab = pd.DataFrame({'topic':[int(t) for t in CORE_TOPICS], 'label':[TOPIC_LABEL[int(t)] for t in CORE_TOPICS]})
core_lab.to_csv(RESULT_DIR / f'핵심의제_라벨_{PERIOD}.csv', index=False, encoding='utf-8-sig')
print('저장: 핵심의제_라벨(04와 공유)')


## 검증 체크리스트
- 의제비중 주지표 = soft topic mass(theta 평균) + 문서 부트스트랩 CI(press 고정·내부 문서 재표집), 분모=우주 N
- 비중표 2버전(절대/조건부), dominant 비율·그룹별 uncertain율은 보조
- log-odds는 같은 핵심의제 내부·셀=그룹×의제·이중기준(≥50문서 AND ≥5,000토큰) 발동 셀만, z-score 주지표 → 셀발동표로 확인
- log-odds는 본문서 보조적 어휘분석으로 격하(주증거는 막대그래프 topic mass)
- 강건성 패널 = 개별 매체 점(CI 아님), 그룹당 매체 2~3개 한계 명시
- 그래프 한글 정상 렌더(폰트 폴백)
